# I. Problem Framing
### Project title
**Forecasting Hourly Bike-Sharing Demand for Sustainable Urban Mobility**

### Problem statement
Urban bike-sharing systems need accurate short-term demand forecasts so operators can place bikes where they are needed most, reduce shortages, and improve service reliability. In this project, I will build a regression model to predict the **hourly number of bike** rentals using weather, seasonal, and calendar features from the UCI Bike Sharing dataset.

### Why this matters
Bike-sharing is an important part of sustainable transportation because it can help reduce traffic congestion, lower emissions, and improve last-mile connectivity. Better demand forecasting can support more efficient bike redistribution, fewer empty docks, and a better rider experience, which may encourage greater use of shared cycling instead of private cars.

### Stakeholders
This project is relevant to:

* **Bike-sharing operators**, who need to manage fleet distribution efficiently.

* **City transport planners**, who want to improve mobility options.

* **Sustainability teams**, who are interested in low-carbon transport solutions.


### Project objective
The objective of this project is to:

1. Predict hourly bike rental demand as accurately as possible.

2. Compare multiple regression models.

3. Identify the most important factors affecting demand.

4. Translate the results into practical recommendations for operational planning.

# II. Hypothesis
I believe that hour of the day (`hr`) and weather conditions (`weathersit`, `temp`, `hum`, `windspeed`) influence hourly bike rental demand (`cnt`), such that demand is highest during peak commuting hours on clear, mild days and lowest during extreme weather or late‑night hours.

I also hypothesize that a tree-based model will outperform a simple linear model because bike demand is likely affected by nonlinear relationships and feature interactions.

# III. Dataset Description & Rationale
### 1. Dataset description and source

I selected the **Bike Sharing dataset (hourly)** from the UCI Machine Learning Repository.

🔗 Source: https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset

The dataset contains hourly records of bike rentals from Capital Bikeshare in Washington, D.C., between 2011 and 2012. Each row represents one hour, with information about the date, season, weather conditions, and the total number of bikes rented in that hour (`cnt`).

### 2. Justification for selection

**Relevance to the regression task**

* The dataset is explicitly designed for regression since the primary goal is to predict the continuous target variable cnt (total bike rentals per hour).

* It includes a rich set of numerical and categorical features (`hr`, `season`, `weathersit`, `temp`, `hum`, `windspeed`, `weekday`, `workingday`, etc.) that can be used to capture temporal patterns, weather‑related demand shifts, and weekday/weekend effects.

* This structure supports a clear regression workflow: feature engineering, model training, and evaluation using RMSE, MAE, and R².

### 3. Target and features available

**Target:** `cnt` — the total number of rental bikes in the hour (including both casual and registered users).

**Features:**

* __Time‑based:__ `hr`, `season`, `yr`, `mnth`, `weekday`, `workingday`, `holiday`.

* __Weather‑based:__ `weathersit`, `temp`, `atemp`, `hum`, `windspeed`.

These features allow me to explore nonlinear and interaction‑rich relationships between time‑of‑day, weather, and demand, which is a good fit for comparing linear models and tree‑based models.

### 4. Cleanliness and licensing

* The dataset is relatively clean: it is provided as a single CSV file with clear column descriptions, no missing values, and consistent encoding.

* The data is **well‑documented**, with a short description of each variable and its meaning, which makes preprocessing and interpretation straightforward.

* The dataset is licensed under the **Creative Commons Attribution 4.0 International (CC BY 4.0)** license.

  * This allows me to share, adapt, and use the dataset for any purpose, including this capstone project, as long as I give appropriate credit to the original authors and to UCI.


### 5. Sustainability impact

* Bike‑sharing systems are a key component of __sustainable urban mobility__ because they reduce reliance on private cars, lower traffic congestion, and cut greenhouse gas emissions.

* By predicting hourly demand, this project can help __bike‑share operators and city planners__ optimize fleet allocation, reduce empty docks and overfilled stations, and improve service reliability.

* More accurate demand forecasts can support __larger adoption of shared cycling__, encouraging citizens to choose low‑carbon transport options and helping cities meet sustainability and climate‑action targets.

# IV. Data Preparation & EDA

**Workflow:**
1. Load raw data

2. Remove duplicates and fix obvious errors (structural)

3. Split into train/test sets

4. Calculate statistics on training set only

5. Apply statistical transformations using training parameters

In [38]:
import pandas as pd

# Load dataset
df = pd.read_csv('../data/raw/hourly_bikesharing_2011_2012.csv')
print(df.head(5))

# Check for exact duplicates across all columns
exact_duplicates = df.duplicated().sum()
print(f"Exact duplicates found: {exact_duplicates}")

# View duplicate rows
duplicate_rows = df[df.duplicated(keep=False)]
print(f"Total rows involved in duplication: {len(duplicate_rows)}")

   instant      dteday  season  yr  mnth  hr  holiday  weekday  workingday  \
0        1  2011-01-01       1   0     1   0        0        6           0   
1        2  2011-01-01       1   0     1   1        0        6           0   
2        3  2011-01-01       1   0     1   2        0        6           0   
3        4  2011-01-01       1   0     1   3        0        6           0   
4        5  2011-01-01       1   0     1   4        0        6           0   

   weathersit  temp   atemp   hum  windspeed  casual  registered  cnt  
0           1  0.24  0.2879  0.81        0.0       3          13   16  
1           1  0.22  0.2727  0.80        0.0       8          32   40  
2           1  0.22  0.2727  0.80        0.0       5          27   32  
3           1  0.24  0.2879  0.75        0.0       3          10   13  
4           1  0.24  0.2879  0.75        0.0       0           1    1  
Exact duplicates found: 0
Total rows involved in duplication: 0


In [28]:
print("=== 1. Impossible / domain‑invalid values ===")

# 1.1 Impossible values: negative rentals or rentals that are not non‑negative integers
for col in ["casual", "registered", "cnt"]:
    neg = (df[col] < 0).sum()
    print(f"Negative values in {col}: {neg}")

# 1.2 Impossible values: hour not in 0–23
invalid_hr = (~df["hr"].between(0, 23)).sum()
print(f"Hours outside 0–23: {invalid_hr}")

# 1.3 Impossible values: season not 1–4
invalid_season = (~df["season"].isin([1, 2, 3, 4])).sum()
print(f"Seasons not in {1,2,3,4}: {invalid_season}")

# 1.4 Impossible values: holiday, workingday, weekday
# assume they should be small integers (0,1) or 1–7 (weekday)
invalid_holiday = (~df["holiday"].isin([0, 1])).sum()
print(f"Holiday not 0 or 1: {invalid_holiday}")

invalid_workingday = (~df["workingday"].isin([0, 1])).sum()
print(f"Workingday not 0 or 1: {invalid_workingday}")

invalid_weekday = (~df["weekday"].between(0, 6)).sum()
print(f"Weekday not in 0–6: {invalid_weekday}")

# 1.5 Impossible values: weathersit not 1–4
invalid_weathersit = (~df["weathersit"].between(1, 4)).sum()
print(f"Weathersit not 1–4: {invalid_weathersit}")

# 1.6 Impossible values: normalized temp and atemp
print(f"Temperature range: {df['temp'].min():.3f} – {df['temp'].max():.3f}")
print(f"Feeling temp (atemp) range: {df['atemp'].min():.3f} – {df['atemp'].max():.3f}")


print("\n=== 2. Future dates ===")

df["dteday"] = pd.to_datetime(df["dteday"])
print("Date range:", df["dteday"].min(), "to", df["dteday"].max())

# Check if any dates are outside the expected 2011–2012 window
outside_2011_2012 = df[~df["dteday"].dt.year.isin([2011, 2012])]
print(f"Records outside 2011–2012: {len(outside_2011_2012)}")


print("\n=== 3. Wrong units / range‑based checks ===")

# 3.1 Check if normalized temp and atemp are in [0,1] (as described in UCI metadata)
print("Normalized temp outside [0,1]:")
print("Min temp:", df["temp"].min())
print("Max temp:", df["temp"].max())

print("Normalized atemp outside [0,1]:")
print("Min atemp:", df["atemp"].min())
print("Max atemp:", df["atemp"].max())

# 3.2 Check humidity: expect 0–1 after normalization (not 0–100)
print("Humidity range (0–1):", df["hum"].min(), "–", df["hum"].max())

# 3.3 Check windspeed: should be 0–1 (since it is divided by 67)
print("Windspeed range (0–1):", df["windspeed"].min(), "–", df["windspeed"].max())

=== 1. Impossible / domain‑invalid values ===
Negative values in casual: 0
Negative values in registered: 0
Negative values in cnt: 0
Hours outside 0–23: 0
Seasons not in (1, 2, 3, 4): 0
Holiday not 0 or 1: 0
Workingday not 0 or 1: 0
Weekday not in 0–6: 0
Weathersit not 1–4: 0
Temperature range: 0.020 – 1.000
Feeling temp (atemp) range: 0.000 – 1.000

=== 2. Future dates ===
Date range: 2011-01-01 00:00:00 to 2012-12-31 00:00:00
Records outside 2011–2012: 0

=== 3. Wrong units / range‑based checks ===
Normalized temp outside [0,1]:
Min temp: 0.02
Max temp: 1.0
Normalized atemp outside [0,1]:
Min atemp: 0.0
Max atemp: 1.0
Humidity range (0–1): 0.0 – 1.0
Windspeed range (0–1): 0.0 – 0.8507


In [27]:
print("\n=== 4. Format inconsistencies ===")
# 4.1 Confirm dteday is now a proper datetime
print("dteday dtype:", df["dteday"].dtype)

# 4.2 Check for any non‑numeric columns that should be numeric (e.g., temp, hum, windspeed)
numeric_cols = ["temp", "atemp", "hum", "windspeed"]
for col in numeric_cols:
    if df[col].apply(lambda x: isinstance(x, (int, float))).all():
        print(f"{col} is fully numeric")
    else:
        print(f"{col} may have non‑numeric entries")

# 4.3 Quick check: are season, weekday, etc. integers?
for col in ["season", "holiday", "workingday", "weekday", "weathersit"]:
    print(col, "unique values:", sorted(df[col].unique().tolist()))
    

print("\n=== 5. Encoding / corrupted text ===")
# 5.1 Check column names for strange characters
print("Column names:", list(df.columns))

# 5.2 Check for any exotic unicode or non‑ASCII characters in string‑like fields
string_cols = [col for col in df.select_dtypes(include=["object"]).columns]
for col in string_cols:
    sample = df[col].dropna().astype(str).head(10)
    bad_chars = sample.str.contains(r"[^\x00-\x7F]", regex=True).any()
    print(f"Non‑ASCII characters in {col} (sample)?", bad_chars)

# 5.3 Example: look for any obviously corrupted rows (e.g., question marks, garbage)
for col in df.columns:
    if df[col].dtype == "object":
        invalid_mask = df[col].str.contains("�|\\?|\\*", na=False)
        n_bad = invalid_mask.sum()
        if n_bad > 0:
            print(f"Potentially corrupted text in {col}: {n_bad} rows")


=== 4. Format inconsistencies ===
dteday dtype: datetime64[ns]
temp is fully numeric
atemp is fully numeric
hum is fully numeric
windspeed is fully numeric
season unique values: [1, 2, 3, 4]
holiday unique values: [0, 1]
workingday unique values: [0, 1]
weekday unique values: [0, 1, 2, 3, 4, 5, 6]
weathersit unique values: [1, 2, 3, 4]

=== 5. Encoding / corrupted text ===
Column names: ['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']


### 🔎 Data Cleaning Initial Findings
The initial data quality checks revealed no major issues in the Bike Sharing dataset. All numeric fields fell within expected domain ranges, the date column covered only the intended 2011–2012 period, and categorical variables contained only valid encoded values. This suggests the dataset is clean and suitable for regression modeling with minimal preprocessing beyond feature selection and encoding.


**Next step:**
Because the bike-sharing dataset is time-ordered, I will use a temporal split rather than a random split. This ensures that the model is evaluated on future, unseen hours, which better reflects the real forecasting use case and avoids data leakage from the future into training.

In [39]:
def temporal_split(df, date_column, split_date):
    """Split dataset chronologically"""
    
    df[date_column] = pd.to_datetime(df[date_column])
    df_sorted = df.sort_values(date_column)
    
    train_data = df_sorted[df_sorted[date_column] < split_date].copy()
    test_data = df_sorted[df_sorted[date_column] >= split_date].copy()
    
    return train_data, test_data


split_date = "2012-12-01"
train_df, test_df = temporal_split(df, "dteday", split_date)

print(f"\nTemporal split at {split_date}:")
print(f"Training period: {train_df['dteday'].min()} to {train_df['dteday'].max()}")
print(f"Test period: {test_df['dteday'].min()} to {test_df['dteday'].max()}")
print(f"Train size: {len(train_df)} rows")
print(f"Test size: {len(test_df)} rows")


Temporal split at 2012-12-01:
Training period: 2011-01-01 00:00:00 to 2012-11-30 00:00:00
Test period: 2012-12-01 00:00:00 to 2012-12-31 00:00:00
Train size: 16637 rows
Test size: 742 rows


# V. Modeling & Evaluation

# VI. Limitations, Ethics & Impact